# Advanced Analytics + Risk Metrics
**Bluestock Fintech Internship — Day 5**

Tasks covered:
1. Historical VaR (95%) & CVaR — all 40 schemes
2. Rolling 90-day Sharpe — top 5 funds
3. Investor cohort analysis
4. SIP continuity analysis
5. Fund recommender by risk appetite
6. Sector HHI concentration (equity funds)

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')

BASE = os.getcwd()
PROC = os.path.join(BASE, 'data', 'processed')
DASH = os.path.join(BASE, 'dashboard')

BG='#0d1117'; PANEL='#161b22'; BORDER='#30363d'
BLUE='#4fc3f7'; GREEN='#56d364'; AMBER='#e3b341'
RED='#f85149'; PURPLE='#bc8cff'; WHITE='#e6edf3'; GRAY='#8b949e'
ACCENT=[BLUE,GREEN,AMBER,RED,PURPLE,'#ff7b72','#79c0ff','#ffa657']

nav = pd.read_csv(os.path.join(PROC,'nav_history_clean.csv'), parse_dates=['date'])
fm  = pd.read_csv(os.path.join(PROC,'fund_master_clean.csv'))
txn = pd.read_csv(os.path.join(PROC,'investor_transactions_clean.csv'), parse_dates=['txn_date'])
ph  = pd.read_csv(os.path.join(PROC,'portfolio_holdings_clean.csv'))
sc  = pd.read_csv(os.path.join(BASE,'fund_scorecard.csv'), index_col=0)

nav = nav.sort_values(['scheme_code','date']).reset_index(drop=True)
names   = fm.set_index('scheme_code')['scheme_name'].to_dict()
rg_map  = fm.set_index('scheme_code')['risk_grade'].to_dict()
nav['daily_ret'] = nav.groupby('scheme_code')['nav'].pct_change()
nav = nav.dropna(subset=['daily_ret'])
RF = 0.065 / 252
print('Data loaded. Schemes:', nav['scheme_code'].nunique())

## Task 1 — Historical VaR (95%) & CVaR
- **VaR 95%** = 5th percentile of daily return distribution
- **CVaR 95%** = mean of all returns below the VaR threshold (expected shortfall)
- Computed for all 40 schemes

In [ ]:
rows = []
for code, grp in nav.groupby('scheme_code'):
    rets = grp['daily_ret'].dropna()
    var95  = np.percentile(rets, 5)
    cvar95 = rets[rets <= var95].mean()
    rows.append({
        'scheme_code': code,
        'scheme_name': names.get(code,'')[:40],
        'risk_grade':  rg_map.get(code,''),
        'var_95_daily_pct':  round(var95*100, 4),
        'cvar_95_daily_pct': round(cvar95*100, 4),
        'ann_return_pct':    round(rets.mean()*252*100, 2),
        'ann_vol_pct':       round(rets.std()*np.sqrt(252)*100, 2),
    })

var_df = pd.DataFrame(rows).sort_values('var_95_daily_pct')
var_df.to_csv(os.path.join(BASE,'var_cvar_report.csv'), index=False)
print('Saved: var_cvar_report.csv')
var_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
for ax in axes:
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)

top10_var = var_df.head(10)
short = [n[:25] for n in top10_var['scheme_name']]
axes[0].barh(short, top10_var['var_95_daily_pct'], color=RED, alpha=0.85)
axes[0].set_title('Top 10 Highest VaR (95%) Funds', color=BLUE, fontsize=11)
axes[0].set_xlabel('VaR Daily %', color=GRAY)
axes[0].tick_params(axis='y', labelsize=7)

axes[1].scatter(var_df['ann_vol_pct'], var_df['var_95_daily_pct'],
                color=AMBER, alpha=0.8, edgecolors=BORDER, s=60)
axes[1].set_title('Volatility vs VaR', color=BLUE, fontsize=11)
axes[1].set_xlabel('Ann. Volatility (%)', color=GRAY)
axes[1].set_ylabel('VaR 95% Daily (%)', color=GRAY)

fig.patch.set_facecolor(BG)
plt.tight_layout()
plt.show()

## Task 2 — Rolling 90-Day Sharpe Ratio

In [ ]:
top5 = sc.head(5)['scheme_code'].tolist()

fig, ax = plt.subplots(figsize=(14, 6), facecolor=BG)
ax.set_facecolor(PANEL)
ax.tick_params(colors=GRAY, labelsize=8)
for sp in ax.spines.values(): sp.set_edgecolor(BORDER)

for i, code in enumerate(top5):
    grp = nav[nav['scheme_code']==code].set_index('date')['daily_ret']
    roll = ((grp.rolling(90).mean() - RF) / grp.rolling(90).std()) * np.sqrt(252)
    label = names.get(code,'').replace(' Direct Growth','').replace(' Mutual Fund','')[:28]
    ax.plot(roll.index, roll.values, color=ACCENT[i], linewidth=1.5, label=label)

ax.axhline(0, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Rolling 90-Day Sharpe Ratio — Top 5 Funds', color=BLUE, fontsize=13)
ax.set_ylabel('Sharpe Ratio', color=GRAY)
ax.legend(fontsize=8, facecolor=PANEL, labelcolor=WHITE)
fig.patch.set_facecolor(BG)
plt.tight_layout()
plt.savefig(os.path.join(DASH,'rolling_sharpe_chart.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: rolling_sharpe_chart.png')

## Task 3 — Investor Cohort Analysis

In [ ]:
sip = txn[txn['txn_type']=='SIP'].copy()
first_yr = sip.groupby('investor_id')['txn_date'].min().dt.year.rename('cohort_year')
sip = sip.join(first_yr, on='investor_id')

cohort = sip.groupby('cohort_year').agg(
    investors      = ('investor_id','nunique'),
    avg_sip_amt    = ('amount','mean'),
    total_invested = ('amount','sum'),
).round(2)

top_fund = (
    sip.groupby(['cohort_year','scheme_code'])['amount'].sum().reset_index()
    .sort_values('amount', ascending=False)
    .groupby('cohort_year').first()['scheme_code']
    .map(lambda c: names.get(c,'')[:30]).rename('top_fund')
)
cohort = cohort.join(top_fund)
cohort.to_csv(os.path.join(BASE,'cohort_analysis.csv'))
print('Saved: cohort_analysis.csv')
cohort

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG)
for ax in axes:
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)

axes[0].bar(cohort.index.astype(str), cohort['total_invested']/1e5, color=BLUE, alpha=0.85)
axes[0].set_title('Total SIP Invested by Cohort (Lakhs)', color=BLUE, fontsize=11)
axes[0].set_xlabel('Cohort Year', color=GRAY)
axes[0].set_ylabel('Amount (Lakhs)', color=GRAY)

axes[1].bar(cohort.index.astype(str), cohort['avg_sip_amt'], color=GREEN, alpha=0.85)
axes[1].set_title('Avg SIP Amount by Cohort', color=BLUE, fontsize=11)
axes[1].set_xlabel('Cohort Year', color=GRAY)
axes[1].set_ylabel('Avg Amount (Rs)', color=GRAY)

fig.patch.set_facecolor(BG)
plt.tight_layout()
plt.show()

## Task 4 — SIP Continuity Analysis

In [ ]:
sip_s = sip.sort_values(['folio_no','txn_date'])
sip_s['prev_date'] = sip_s.groupby('folio_no')['txn_date'].shift(1)
sip_s['gap_days']  = (sip_s['txn_date'] - sip_s['prev_date']).dt.days

freq   = sip_s.groupby('folio_no')['txn_date'].count()
active = freq[freq >= 3].index
sip_a  = sip_s[sip_s['folio_no'].isin(active)]

cont = sip_a.groupby('folio_no')['gap_days'].mean().reset_index()
cont.columns = ['folio_no','avg_gap_days']
cont['at_risk'] = cont['avg_gap_days'] > 35
cont.to_csv(os.path.join(BASE,'sip_continuity.csv'), index=False)

at_risk = cont['at_risk'].sum()
total   = len(cont)
rate    = round((1 - at_risk/total)*100, 1) if total > 0 else 0

print(f'Active folios (3+ SIPs): {total}')
print(f'At-risk (gap > 35 days): {at_risk}')
print(f'SIP continuity rate:     {rate}%')
cont.head(10)

## Task 5 — Fund Recommender by Risk Appetite

In [ ]:
risk_map = {
    'Low':      ['Low'],
    'Moderate': ['Moderate','Moderately High'],
    'High':     ['High','Very High','Moderately High'],
}
sc['risk_grade'] = sc['scheme_code'].map(rg_map)

def recommend(appetite):
    grades   = risk_map[appetite]
    filtered = sc[sc['risk_grade'].isin(grades)].nlargest(3,'sharpe')
    out = filtered[['scheme_name','risk_grade','sharpe','cagr_3y','alpha_annual','score']].copy()
    out.columns = ['Fund','Risk Grade','Sharpe','CAGR 3Y','Alpha %','Score']
    out['Fund'] = out['Fund'].str.replace(' Direct Growth','').str.replace(' Mutual Fund','').str[:35]
    return out.reset_index(drop=True)

for appetite in ['Low','Moderate','High']:
    print(f'\n--- {appetite} Risk ---')
    display(recommend(appetite))

## Task 6 — Sector HHI Concentration

In [ ]:
equity_codes = fm[fm['category']=='Equity']['scheme_code'].tolist()
ph_eq = ph[ph['scheme_code'].isin(equity_codes)].copy()

hhi_rows = []
for code, grp in ph_eq.groupby('scheme_code'):
    w   = grp['weight_pct'] / 100
    hhi = (w**2).sum()
    top_sec = grp.groupby('sector')['weight_pct'].sum().idxmax()
    hhi_rows.append({
        'scheme_code':   code,
        'scheme_name':   names.get(code,'')[:35],
        'hhi':           round(hhi, 4),
        'concentration': 'High' if hhi>0.15 else 'Moderate' if hhi>0.08 else 'Low',
        'top_sector':    top_sec,
    })

hhi_df = pd.DataFrame(hhi_rows).sort_values('hhi', ascending=False)
hhi_df.to_csv(os.path.join(BASE,'hhi_concentration.csv'), index=False)
print('Saved: hhi_concentration.csv')
hhi_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
ax.set_facecolor(PANEL)
ax.tick_params(colors=GRAY, labelsize=8)
for sp in ax.spines.values(): sp.set_edgecolor(BORDER)

colors_hhi = [RED if h>0.15 else AMBER if h>0.08 else GREEN for h in hhi_df['hhi']]
short = [n[:25] for n in hhi_df['scheme_name']]
ax.barh(short, hhi_df['hhi'], color=colors_hhi, alpha=0.85)
ax.axvline(0.15, color=RED,   linewidth=1, linestyle='--', label='High (>0.15)')
ax.axvline(0.08, color=AMBER, linewidth=1, linestyle='--', label='Moderate (>0.08)')
ax.set_title('Sector HHI Concentration — Equity Funds', color=BLUE, fontsize=12)
ax.set_xlabel('HHI Score', color=GRAY)
ax.legend(fontsize=8, facecolor=PANEL, labelcolor=WHITE)
fig.patch.set_facecolor(BG)
plt.tight_layout()
plt.show()

---
## 5 Advanced Insights

### Insight 1 — Funds with Highest VaR Risk
**Nippon India Flexi Cap Fund** has the highest daily VaR at **-0.38%**, meaning on a bad day (5th percentile), investors can expect to lose 0.38% of their investment. Its CVaR is even worse at ~-0.55%, indicating heavy tail risk. In contrast, **Aditya Birla Flexi Cap Fund** has the lowest VaR at -0.14%, making it the safest from a downside risk perspective. Funds with High/Very High risk grades consistently show VaR below -0.30%, confirming the risk grade labels are meaningful.

### Insight 2 — Investor Cohorts: Who Invests the Most?
The **2023 cohort** has the highest total SIP investment at **Rs 27.45 Lakh** with an average SIP of Rs 16,948 — the highest across all cohorts. This suggests newer investors are committing larger amounts per SIP, possibly driven by increased financial awareness. The **2022 cohort** has the lowest avg SIP (Rs 11,874), possibly reflecting market uncertainty during that period. The top fund preference shifted from DSP Flexi Cap (2020) to ICICI Short Duration (2021–22) to Nippon Flexi Cap (2023), showing evolving investor preferences.

### Insight 3 — SIP Continuity Rate
Among folios with 3+ SIP transactions, the continuity analysis shows the average gap between SIP dates. Folios flagged as **at-risk** (avg gap > 35 days) indicate investors who are skipping monthly SIPs — a key churn signal. A high continuity rate (>80%) means most investors are disciplined. At-risk investors should be targeted with nudge campaigns to resume their SIPs before they lapse entirely.

### Insight 4 — Rolling Sharpe Reveals Market Cycles
The rolling 90-day Sharpe chart shows that **all top 5 funds had negative Sharpe ratios during 2022** (market correction period), confirming systematic risk. **Quantum Balanced Advantage** is the only fund that briefly crossed into positive Sharpe territory in 2023–24, driven by its defensive allocation strategy. Funds with consistently negative rolling Sharpe (below -1.0) during downturns are poor risk-adjusted performers despite high absolute returns.

### Insight 5 — Sector Concentration Risk (HHI)
**Kotak Mahindra Mid Cap Fund** has the highest HHI of **0.22** (High concentration) with Banking as the dominant sector. This means ~22% of portfolio variance comes from a single sector bet. **HDFC Large Cap** and **HDFC ELSS** also show High HHI (>0.15), concentrated in Auto and FMCG respectively. Investors seeking diversification should prefer funds with HHI < 0.08 (Low concentration). High HHI funds can outperform in bull markets for that sector but carry significant drawdown risk during sector-specific downturns.

---
## Deliverables Summary

| File | Description |
|------|-------------|
| `var_cvar_report.csv` | VaR & CVaR for all 40 schemes |
| `cohort_analysis.csv` | Investor cohort metrics by first SIP year |
| `sip_continuity.csv` | Folio-level SIP gap analysis + at-risk flag |
| `hhi_concentration.csv` | Sector HHI for all equity funds |
| `recommender.py` | Standalone fund recommender script |
| `dashboard/rolling_sharpe_chart.png` | Rolling 90-day Sharpe chart |